# Example: accessing the converted datasets

All datasets share one format (see `README.md`): three tables per dataset (`subjects`, `records`, `bp`) and one HDF5 wave file per record. Convert first with `uv run python scripts/structure.py`.

In [ ]:
import ppg_bp_database as db

db.summary()

## Tables
Each `load_*` function takes one dataset name, a list of names, or nothing (= all converted datasets).

The `synthetic` column of `subjects` and `records` separates simulated data (`pwdb`) from measured data, e.g. `records[~records["synthetic"]]`.

In [ ]:
subjects = db.load_subjects()
records = db.load_records()
bp = db.load_bp()
records.head()

In [ ]:
# join everything into one table: one row per BP value, with record and subject info
table = bp.merge(records, on="record_id").merge(subjects, on=["subject_id", "dataset", "synthetic"])
table[["record_id", "dataset", "synthetic", "condition", "age", "sex", "time_s", "sbp", "dbp"]].head()

## Waves
`load_wave` takes one row of the records table and returns `{signal name: pd.Series indexed by time in s}`. `channels`, `start_s` and `end_s` load only a part of the file.

In [ ]:
record = records[records["dataset"] == "pttppg"].iloc[0]
waves = db.load_wave(record, channels=["ecg", "ppg_1"], start_s=10, end_s=15)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(waves), 1, sharex=True, figsize=(10, 4))
for ax, (name, s) in zip(axes, waves.items()):
    ax.plot(s.index, s.values)
    ax.set_ylabel(name)
axes[-1].set_xlabel("time (s)")
fig.suptitle(record["record_id"]);

In [ ]:
# BP values of that record, on the same time axis as the waves
bp[bp["record_id"] == record["record_id"]]